### Vacabulary building and True Ingredient Matching

In [2]:
import json
import pandas as pd
from pathlib import Path

# Load training + validation subsets
with open("data/mini_data/recipes.json", "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open("data/mini_data_val/recipes.json", "r", encoding="utf-8") as f:
    val_data = json.load(f)

# Combine and label the source
train_df = pd.DataFrame(train_data)
val_df = pd.DataFrame(val_data)
train_df["split"] = "train"
val_df["split"] = "val"
recipe_df = pd.concat([train_df, val_df], ignore_index=True)

# Strip .jpg for merging
recipe_df["image_id"] = recipe_df["image_id"].str.replace(".jpg", "", regex=False)

# Load layer2.json
with open("data/recipe1m_images/layer2.json", "r", encoding="utf-8") as f:
    layer2 = json.load(f)

# Map image_id to recipe ID using layer2
image_to_recipe = {}
for entry in layer2:
    recipe_id = entry["id"]
    for img in entry.get("images", []):
        img_id = img.get("id", "").replace(".jpg", "")
        image_to_recipe[img_id] = recipe_id

# Add recipe ID column to combined subset based on image ID
recipe_df["recipe_id"] = recipe_df["image_id"].map(image_to_recipe)

# Load det_ingrs.json
with open("data/recipe1m_images/det_ingrs.json", "r", encoding="utf-8") as f:
    det_data = json.load(f)
det_df = pd.DataFrame(det_data)

# Extract detected ingredients if needed
if "detected_ingredients" not in det_df.columns:
    det_df["detected_ingredients"] = det_df.apply(
        lambda row: [i["text"] for i, v in zip(row["ingredients"], row["valid"]) if v], axis=1
    )

# Merge on recipe ID
merged_df = pd.merge(
    recipe_df,
    det_df[["id", "detected_ingredients"]],
    how="inner",
    left_on="recipe_id",
    right_on="id"
)

print(f"Merged {len(merged_df)} entries (train + val)")
print(merged_df[["split", "image_id", "ingredients", "detected_ingredients"]].head())
merged_df.to_json("data/mini_data/recipes_with_detected.json", orient="records", indent=2)


Merged 11000 entries (train + val)
   split    image_id                                        ingredients  \
0  train  177daaccae  [1 tablespoon butter, 1- 1/2 cup kimchi, cut i...   
1  train  af20e0e0df  [1 lb ground lean pork or 1 lb ground round, 1...   
2  train  715316e229  [1 cup bulgar wheat, 1 12-2 cups boiling water...   
3  train  9bc62c4488  [1 whole onion, diced, 1 whole red pepper, see...   
4  train  44d6065bb3  [2 cans trappeys black-eyed peas, 1 can petite...   

                                detected_ingredients  
0  [butter, tuna, water, broth, sea salt, green o...  
1  [ground lean pork, green onions, water chestnu...  
2  [bulgur wheat, boiling water, chickpeas, mint,...  
3  [onion, red pepper, ground beef, oil, eggs, milk]  
4  [jalapeno peppers, small onion, yellow bell pe...  


In [3]:
import json
from collections import Counter
from pathlib import Path
import pickle
import pandas as pd
import os

# Load recipes
with open("data/mini_data/recipes.json", "r", encoding="utf-8") as f:
    recipes = json.load(f)

# Build vocab from all ingredient words
counter = Counter()
for r in recipes:
    for ing in r["ingredients"]:
        tokens = ing.lower().split()
        counter.update(tokens)

# Keep words with at least 2 occurrences
min_freq = 2
vocab = ["<pad>", "<unk>"]
vocab += [word for word, freq in counter.items() if freq >= min_freq]

word2idx = {w: i for i, w in enumerate(vocab)}

# Save vocab
with open("data/mini_data/vocab.pkl", "wb") as f:
    pickle.dump(word2idx, f)

print(f"Vocab size: {len(word2idx)}")


Vocab size: 6021


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import json, pickle, random
from pathlib import Path
from tqdm import tqdm
import os
from multimodal_encoder import ImageEncoder, JointEncoderWithCrossAttention, BertIngredientEncoder

# === Config ===
BATCH_SIZE = 32
EMBED_DIM = 512
MAX_LEN = 20
EPOCHS = 1
LR = 1e-4
CHECKPOINT_PATH = "checkpoints"
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

# === Dataset ===
class JointDataset(torch.utils.data.Dataset):
    def __init__(self, json_path, image_dir, word2idx, transform, max_len=20):
        with open(json_path, "r") as f:
            self.data = json.load(f)
        self.image_dir = Path(image_dir)
        self.word2idx = word2idx
        self.transform = transform
        self.max_len = max_len

    def tokenize(self, tokens):
        ids = []
        for txt in tokens:
            for tok in txt.lower().split():
                ids.append(self.word2idx.get(tok, self.word2idx["<unk>"]))
        ids = ids[:self.max_len] + [self.word2idx["<pad>"]] * (self.max_len - len(ids))
        return ids

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        anchor = self.data[idx]
        image = self.transform(Image.open(self.image_dir / anchor["image"]).convert("RGB"))
        pos = self.tokenize(anchor["ingredients"] + anchor["instructions"])

        neg_idx = random.choice([i for i in range(len(self.data)) if i != idx])
        neg = self.data[neg_idx]
        neg_txt = self.tokenize(neg["ingredients"] + neg["instructions"])

        return image, torch.tensor(pos), torch.tensor(neg_txt)

# === Training ===
def train():
    with open("data/mini_data/vocab.pkl", "rb") as f:
        word2idx = pickle.load(f)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    dataset = JointDataset("data/mini_data/recipes.json", "data/mini_data", word2idx, transform, max_len=MAX_LEN)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    image_encoder = ImageEncoder(
                                embed_dim=EMBED_DIM,
                                model_name='resnet18',        # or 'resnet50', 'vit_b_16', 'efficientnet_b0'
                                freeze_backbone=True,         # freeze everything
                                tune_last_n_blocks=0          # OR set > 0 to enable partial fine-tuning
                            ).cuda()
    text_encoder = JointEncoderWithCrossAttention(vocab_size=len(word2idx), embed_dim=EMBED_DIM).cuda()

    params = list(image_encoder.parameters()) + list(text_encoder.parameters())
    optimizer = torch.optim.Adam(params, lr=LR)

    def triplet_loss(a, p, n, margin=0.2):
        return torch.clamp(margin + F.cosine_similarity(a, n) - F.cosine_similarity(a, p), min=0).mean()
    
    print(f"Backbone frozen: {image_encoder.frozen}")
    
    for epoch in range(EPOCHS):
        image_encoder.train()
        text_encoder.train()
        total_loss = 0

        for img, pos, neg in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
            img, pos, neg = img.cuda(), pos.cuda(), neg.cuda()
            img_emb = image_encoder(img)
            pos_emb = text_encoder(img_emb, pos)
            neg_emb = text_encoder(img_emb, neg)

            loss = triplet_loss(img_emb, pos_emb, neg_emb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch+1}: Loss = {total_loss / len(dataloader):.4f}")
        torch.save({
            'image_encoder': image_encoder.state_dict(),
            'text_encoder': text_encoder.state_dict()
        }, Path(CHECKPOINT_PATH) / f"joint_encoder_epoch{epoch+1}.pth")

if __name__ == '__main__':
    train()


C:\Users\Me\Desktop\New folder (2)\7643_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TypeError: ImageEncoder.__init__() got an unexpected keyword argument 'freeze_backbone'

### Dataset

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import json
from pathlib import Path

class RecipeDatasetForEval(Dataset):
    def __init__(self, json_path, image_dir, word2idx, transform, max_len=20, mode='dual'):
        with open(json_path, 'r', encoding='utf-8') as f:
            self.data = json.load(f)
        self.image_dir = Path(image_dir)
        self.word2idx = word2idx
        self.transform = transform
        self.max_len = max_len
        self.mode = mode

    def tokenize(self, entries):
        ids = []
        for e in entries:
            for tok in e.lower().split():
                ids.append(self.word2idx.get(tok, self.word2idx['<unk>']))
        ids = ids[:self.max_len] + [self.word2idx['<pad>']] * (self.max_len - len(ids))
        return torch.tensor(ids)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image_path = self.image_dir / item['image']
        image = self.transform(Image.open(image_path).convert('RGB'))

        if self.mode == 'dual':
            tokens = self.tokenize(item['ingredients'])
            return image, tokens, idx

        elif self.mode == 'joint':
            combined = item['ingredients'] + item['instructions']
            tokens = self.tokenize(combined)
            return image, tokens

        else:
            raise ValueError("Mode must be either 'dual' or 'joint'")

### Compute Recall

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import pairwise_distances
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns

@torch.no_grad()
def compute_recall_at_k_dual(model, dataloader, k_list=[1, 5, 10]):
    model.eval()
    all_img_embs, all_txt_embs = [], []

    for img, txt, _ in tqdm(dataloader, desc="Dual Model Eval"):
        img, txt = img.cuda(), txt.cuda()
        img_emb, txt_emb = model(img, txt)
        all_img_embs.append(img_emb)
        all_txt_embs.append(txt_emb)

    img_matrix = torch.cat(all_img_embs)
    txt_matrix = torch.cat(all_txt_embs)
    sims = img_matrix @ txt_matrix.T
    ranks = sims.argsort(dim=1, descending=True)
    correct = torch.arange(len(ranks)).unsqueeze(1).to(ranks.device)

    return {
    "recalls": {f"Recall@{k}": (ranks[:, :k] == correct).any(dim=1).float().mean().item() for k in k_list},
    "img_embs": img_matrix,
    "txt_embs": txt_matrix
}

@torch.no_grad()
def compute_recall_at_k_joint(image_encoder, text_encoder, dataloader, k_list=[1, 5, 10]):
    image_encoder.eval()
    text_encoder.eval()
    all_img_embs, all_txt_embs = [], []

    for img, txt in tqdm(dataloader, desc="Joint Encoder Eval"):
        img, txt = img.cuda(), txt.cuda()
        img_emb = image_encoder(img)
        txt_emb = text_encoder(img_emb, txt)
        all_img_embs.append(img_emb)
        all_txt_embs.append(txt_emb)

    img_matrix = torch.cat(all_img_embs)
    txt_matrix = torch.cat(all_txt_embs)
    sims = img_matrix @ txt_matrix.T
    ranks = sims.argsort(dim=1, descending=True)
    correct = torch.arange(len(ranks)).unsqueeze(1).to(ranks.device)

    return {
    "recalls": {f"Recall@{k}": (ranks[:, :k] == correct).any(dim=1).float().mean().item() for k in k_list},
    "img_embs": img_matrix,
    "txt_embs": txt_matrix
}

@torch.no_grad()
def visualize_embeddings(img_embs, txt_embs):
    all_embs = torch.cat([img_embs, txt_embs], dim=0).cpu().numpy()
    labels = ["Image"] * img_embs.size(0) + ["Text"] * txt_embs.size(0)
    tsne = TSNE(n_components=2, perplexity=30)
    reduced = tsne.fit_transform(all_embs)
    
    plt.figure(figsize=(10, 7))
    sns.scatterplot(x=reduced[:, 0], y=reduced[:, 1], hue=labels, alpha=0.7)
    plt.title("t-SNE of Joint Embeddings")
    plt.show()

### Train with Hard Negatives + Triplet Loss

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import matplotlib.pyplot as plt
import json, pickle, random
from PIL import Image
from pathlib import Path
import os

from multimodal_encoder import ImageEncoder, JointEncoderWithCrossAttention, TransformerEncoder

# === Config ===
BATCH_SIZE = 32
EPOCHS = 1
EMBED_DIM = 512
LR = 1e-4
CHECKPOINT_PATH = "checkpoints"
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

# === Dataset ===
class HardNegativeDataset(torch.utils.data.Dataset):
    def __init__(self, json_path, image_dir, word2idx, transform, max_len=20, mode="joint"):
        with open(json_path, "r") as f:
            self.data = json.load(f)
        self.image_dir = Path(image_dir)
        self.word2idx = word2idx
        self.transform = transform
        self.max_len = max_len
        self.mode = mode

    def tokenize(self, texts):
        ids = []
        for t in texts:
            for token in t.lower().split():
                ids.append(self.word2idx.get(token, self.word2idx["<unk>"]))
        ids = ids[:self.max_len] + [self.word2idx['<pad>']] * (self.max_len - len(ids))
        return torch.tensor(ids)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        anchor = self.data[idx]
        image = self.transform(Image.open(self.image_dir / anchor["image"]).convert("RGB"))
        if self.mode == "dual":
            pos = self.tokenize(anchor["ingredients"])
            neg_idx = random.choice([i for i in range(len(self.data)) if i != idx])
            neg = self.tokenize(self.data[neg_idx]["ingredients"])
        else:
            pos = self.tokenize(anchor["ingredients"] + anchor["instructions"])
            anchor_ingr = anchor["ingredients"][0].lower().split()[0]
            neg_idx = next((i for i in range(len(self.data)) if i != idx and anchor_ingr in self.data[i]["ingredients"][0].lower()), idx)
            neg = self.tokenize(self.data[neg_idx]["ingredients"] + self.data[neg_idx]["instructions"])
        return image, pos, neg

# === Triplet Loss ===
def triplet_loss(anchor, positive, negative, margin=0.2):
    return torch.clamp(margin + F.cosine_similarity(anchor, negative) - F.cosine_similarity(anchor, positive), min=0).mean()

# === Train Dual or Joint ===
def train(mode="joint"):
    with open("data/mini_data/vocab.pkl", "rb") as f:
        word2idx = pickle.load(f)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    dataset = HardNegativeDataset("data/mini_data/recipes.json", "data/mini_data", word2idx, transform, mode=mode)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

    eval_dataset = RecipeDatasetForEval("data/mini_data_val/recipes.json", "data/mini_data_val", word2idx, transform, mode=mode)
    eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

    image_encoder = ImageEncoder(embed_dim=EMBED_DIM).cuda()
    if mode == "joint":
        text_encoder = JointEncoderWithCrossAttention(vocab_size=len(word2idx), embed_dim=EMBED_DIM).cuda()
    else:
        text_encoder = TransformerEncoder(vocab_size=len(word2idx), embed_dim=EMBED_DIM).cuda()

    optimizer = torch.optim.Adam(list(image_encoder.parameters()) + list(text_encoder.parameters()), lr=LR)

    all_losses = []
    all_recalls = []

    for epoch in range(EPOCHS):
        image_encoder.train()
        text_encoder.train()
        total_loss = 0

        for img, pos, neg in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
            img, pos, neg = img.cuda(), pos.cuda(), neg.cuda()
            img_emb = image_encoder(img)
            if mode == "joint":
                pos_emb = text_encoder(img_emb, pos)
                neg_emb = text_encoder(img_emb, neg)
            else:
                pos_emb = text_encoder(pos)
                neg_emb = text_encoder(neg)

            loss = triplet_loss(img_emb, pos_emb, neg_emb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        all_losses.append(avg_loss)
        print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f}")

        # Evaluate Recall@K
        if mode == "joint":
            recall = compute_recall_at_k_joint(image_encoder, text_encoder, eval_loader)
        else:
            class DualWrapper(torch.nn.Module):
                def __init__(self, img_enc, txt_enc):
                    super().__init__()
                    self.img_enc = img_enc
                    self.txt_enc = txt_enc
                def forward(self, img, txt):
                    return self.img_enc(img), self.txt_enc(txt)
            recall = compute_recall_at_k_dual(DualWrapper(image_encoder, text_encoder), eval_loader)
        
        all_recalls.append(recall)
        print(f"Epoch {epoch+1} Recall@K: {recall}")

        # only print chart of TSNE embedding visualization (can be commented out)
        if (epoch + 1) % 5 == 0:
            if isinstance(recall, dict) and "img_embs" in recall:
                visualize_embeddings(recall["img_embs"], recall["txt_embs"])

        torch.save({
            'image_encoder': image_encoder.state_dict(),
            'text_encoder': text_encoder.state_dict()
        }, Path(CHECKPOINT_PATH) / f"{mode}_encoder_epoch{epoch+1}.pth")

    # === Plot Loss ===
    plt.figure()
    plt.plot(range(1, EPOCHS+1), all_losses, label="Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training Loss Over Epochs")
    plt.legend()
    plt.savefig(f"{mode}_loss_plot.png")

    # === Plot Recall@K ===
    for k in [1, 5, 10]:
        plt.plot([r[f"Recall@{k}"] for r in all_recalls], label=f"Recall@{k}")
    plt.xlabel("Epoch")
    plt.ylabel("Recall")
    plt.title("Recall@K Over Epochs")
    plt.legend()
    plt.savefig(f"{mode}_recall_plot.png")

if __name__ == "__main__":
    train(mode="joint")  # or train(mode="dual")

### Top K Retrieval and storage

In [ ]:
import torch
import json
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
from multimodal_encoder import ImageEncoder, JointEncoderWithCrossAttention, TransformerEncoder
import pickle
from pathlib import Path

@torch.no_grad()
def run_retrieval_and_save_results(
    model_mode="joint",
    checkpoint_path="checkpoints/joint_encoder_epoch1.pth",
    val_json="data/mini_data_val/recipes.json",
    val_img_dir="data/mini_data_val",
    vocab_path="data/mini_data/vocab.pkl",
    outfile="retrieval_results.json",
    topk=5,
    save_failures_only=False,
    failure_output="retrieval_failures.json"
):
    # === Load model ===
    image_encoder = ImageEncoder(embed_dim=512).cuda()

    checkpoint = torch.load(checkpoint_path)
    vocab_size = checkpoint['text_encoder']['text_embed.weight'].size(0)

    if model_mode == "joint":
        text_encoder = JointEncoderWithCrossAttention(vocab_size=vocab_size, embed_dim=512).cuda()
    else:
        text_encoder = TransformerEncoder(vocab_size=vocab_size, embed_dim=512).cuda()

    image_encoder.load_state_dict(checkpoint['image_encoder'])
    text_encoder.load_state_dict(checkpoint['text_encoder'])
    image_encoder.eval()
    text_encoder.eval()

    # === Load data ===
    with open(vocab_path, "rb") as f:
        word2idx = pickle.load(f)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    eval_dataset = RecipeDatasetForEval(
        json_path=val_json,
        image_dir=val_img_dir,
        word2idx=word2idx,
        transform=transform,
        mode=model_mode
    )

    eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

    image_embeddings = []
    text_embeddings = []
    recipe_entries = []
    image_ids = []
    gt_indices = []

    for batch_idx, (images, tokens) in enumerate(tqdm(eval_loader, desc="Encoding")):
        images, tokens = images.cuda(), tokens.cuda()
        img_emb = image_encoder(images)
        if model_mode == "joint":
            txt_emb = text_encoder(img_emb, tokens)
        else:
            txt_emb = text_encoder(tokens)

        image_embeddings.append(img_emb)
        text_embeddings.append(txt_emb)

        for i in range(images.size(0)):
            entry = eval_dataset.data[batch_idx * eval_loader.batch_size + i]
            recipe_entries.append(entry)
            image_ids.append(entry["image"])
            gt_indices.append(batch_idx * eval_loader.batch_size + i)  # assuming aligned order

    image_embeddings = torch.cat(image_embeddings, dim=0)
    text_embeddings = torch.cat(text_embeddings, dim=0)

    sims = image_embeddings @ text_embeddings.T
    values, indices = torch.topk(sims, k=topk, dim=1)

    results = []
    failures = []

    for i, (img_id, top_idx, gt_idx) in enumerate(zip(image_ids, indices, gt_indices)):
        top_recipes = [recipe_entries[j] for j in top_idx.tolist()]
        hit = gt_idx in top_idx.tolist()
        entry = {
            "query_image": img_id,
            "top_recipes": top_recipes,
            "match_found": hit,
            "ground_truth_idx": gt_idx,
            "retrieved_indices": top_idx.tolist()
        }
        results.append(entry)
        if not hit:
            failures.append(entry)

    with open(outfile, "w") as f:
        json.dump(results, f, indent=2)
    print(f"Saved retrieval results to {outfile}")

    if save_failures_only:
        with open(failure_output, "w") as f:
            json.dump(failures, f, indent=2)
        print(f"Saved {len(failures)} failures to {failure_output}")


In [ ]:
from eval_utils import run_retrieval_and_save_results
from multimodal_encoder import ImageEncoder, JointEncoderWithCrossAttention, TransformerEncoder
from torch.utils.data import DataLoader
from torchvision import transforms
import torch, pickle, json


# === Config ===
MODE = "joint"
VAL_JSON = "data/mini_data_val/recipes.json"
VAL_IMG_DIR = "data/mini_data_val"
VOCAB_PATH = "data/mini_data/vocab.pkl"
CHECKPOINT_PATH = f"checkpoints/{MODE}_encoder_epoch1.pth"
TOPK = 5
OUTFILE = "retrieval_results.json"

# === Load Vocab & Transform ===
with open(VOCAB_PATH, "rb") as f:
    word2idx = pickle.load(f)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# === Load Dataset ===
eval_dataset = RecipeDatasetForEval(
    json_path=VAL_JSON,
    image_dir=VAL_IMG_DIR,
    word2idx=word2idx,
    transform=transform,
    mode=MODE
)
eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False)

# === Load Model ===
checkpoint = torch.load(CHECKPOINT_PATH)
vocab_size = checkpoint['text_encoder']['text_embed.weight'].size(0)

image_encoder = ImageEncoder(embed_dim=512).cuda()
if MODE == "joint":
    text_encoder = JointEncoderWithCrossAttention(vocab_size=vocab_size, embed_dim=512).cuda()
else:
    text_encoder = TransformerEncoder(vocab_size=vocab_size, embed_dim=512).cuda()

image_encoder.load_state_dict(checkpoint['image_encoder'])
text_encoder.load_state_dict(checkpoint['text_encoder'])
image_encoder.eval()
text_encoder.eval()

# === Run Retrieval + Save Results ===
run_retrieval_and_save_results(
    image_encoder=image_encoder,
    text_encoder=text_encoder,
    dataloader=eval_loader,
    dataset=eval_dataset,
    mode=MODE,
    topk=TOPK,
    output_path=OUTFILE,
    only_failures=False,     # optional - only save failed queries

)

In [ ]:
# Optional - show failed retrieval


show_retrieval_failures(
    retrieval_path="retrieval_results.json",
    dataset_dir="data/mini_data_val", 
    num_failures=5
)

In [ ]:
# GEMMA prompt for testing/demos/custom prompt runs

import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load Gemma Model
model_name = "google/gemma-2b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
model.eval()

# Generate from Prompt
def generate_instructions(ingredients, title=None, max_length=256):
    if title:
        prompt = f"Recipe Title: {title}\nIngredients: {', '.join(ingredients)}\nInstructions:"
    else:
        prompt = f"Ingredients: {', '.join(ingredients)}\nInstructions:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_length,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        num_return_sequences=1
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Batch Generate from Top-K Retrievals
def generate_from_topk(topk_json_path, output_path="generated_topk.json"):
    with open(topk_json_path, "r") as f:
        topk_data = json.load(f)

    generations = []
    for entry in topk_data:
        query_img = entry["query_image"]
        for i, candidate in enumerate(entry["top_recipes"]):
            ingredients = candidate["ingredients"]
            title = candidate.get("title", "")
            gen = generate_instructions(ingredients, title)
            generations.append({
                "query_image": query_img,
                "rank": i + 1,
                "title": title,
                "ingredients": ingredients,
                "generated_instructions": gen
            })

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(generations, f, indent=2)
    print(f"Saved {len(generations)} generations to {output_path}")

# Run
if __name__ == "__main__":
    generate_from_topk("retrieval_results.json")



### Generation Evaluation and Re-Ranking

In [ ]:
# After gemma_gen_topk is being ran

import json
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import matplotlib.pyplot as plt

# Config
GENERATED_PATH = "generated_topk.json"
GROUNDTRUTH_PATH = "data/mini_data_val/recipes.json"
OUT_PATH = "reranked_generations.json"
BLEU_WEIGHT = (0.25, 0.25, 0.25, 0.25)

# Load
with open(GENERATED_PATH, "r") as f:
    generated = json.load(f)
with open(GROUNDTRUTH_PATH, "r") as f:
    groundtruth = {r["image"]: r["instructions"] for r in json.load(f)}

# Model for cosine similarity
model = SentenceTransformer("all-MiniLM-L6-v2")

results = []
bleu_scorer = SmoothingFunction().method1

bleu_scores = []
cosine_scores = []

for item in tqdm(generated, desc="Evaluating and Reranking"):
    img_id = item["query_image"]
    gt_instr = " ".join(groundtruth.get(img_id, []))
    gt_emb = model.encode(gt_instr, convert_to_tensor=True)

    scored = []
    for gen in item["topk_generations"]:
        gen_text = gen["generated_instructions"]
        bleu = sentence_bleu([gt_instr.split()], gen_text.split(), weights=BLEU_WEIGHT, smoothing_function=bleu_scorer)
        gen_emb = model.encode(gen_text, convert_to_tensor=True)
        cosine = util.cos_sim(gt_emb, gen_emb).item()

        score = 0.5 * bleu + 0.5 * cosine
        gen["bleu"] = bleu
        gen["cosine"] = cosine
        gen["combined_score"] = score
        scored.append(gen)

    scored.sort(key=lambda x: x["combined_score"], reverse=True)
    results.append({
        "query_image": img_id,
        "topk_generations": scored,
        "best_generation": scored[0] if scored else None
    })

    if scored:
        bleu_scores.append(scored[0]["bleu"])
        cosine_scores.append(scored[0]["cosine"])

# Save results
with open(OUT_PATH, "w") as f:
    json.dump(results, f, indent=2)

print(f"Reranked generations with BLEU + Cosine to {OUT_PATH}")

# Summary Stats
avg_bleu = sum(bleu_scores) / len(bleu_scores)
avg_cosine = sum(cosine_scores) / len(cosine_scores)
print(f"\n📉 Average BLEU: {avg_bleu:.4f}")
print(f"📉 Average Cosine Similarity: {avg_cosine:.4f}")

# Visualization
plt.figure(figsize=(8, 5))
plt.plot(bleu_scores, label="BLEU Scores")
plt.plot(cosine_scores, label="Cosine Similarity")
plt.title("Best Generation Scores per Query")
plt.xlabel("Query Index")
plt.ylabel("Score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig("score_plot.png")
plt.show()


# Print Top-5 Samples by BLEU and Cosine
print("\n Top 5 by BLEU Score:")
top_bleu = sorted(results, key=lambda x: x['best_generation']['bleu'], reverse=True)[:5]
for i, item in enumerate(top_bleu):
    b = item['best_generation']
    print(f"{i+1}. {item['query_image']} | BLEU: {b['bleu']:.4f} | Cosine: {b['cosine']:.4f}")
    print(f"   ↪ {b['generated_instructions'][:100]}...")

print("\n Top 5 by Cosine Similarity:")
top_cosine = sorted(results, key=lambda x: x['best_generation']['cosine'], reverse=True)[:5]
for i, item in enumerate(top_cosine):
    b = item['best_generation']
    print(f"{i+1}. {item['query_image']} | Cosine: {b['cosine']:.4f} | BLEU: {b['bleu']:.4f}")
    print(f"   ↪ {b['generated_instructions'][:100]}...")